# Ingest and validate
Use the same catalog and schema as notebook 00. Pandas handles this tiny 610-row validation task; Spark persists and queries the Delta tables. This is a deliberately small hybrid workflow, not a claim of distributed data cleaning.

In [ ]:
import re
# EDIT CATALOG if the detected default is not a writable Unity Catalog catalog.
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "iman_mining_lab"
VOLUME = "project_files"
for identifier in [CATALOG, SCHEMA, VOLUME]:
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier), "Use simple catalog/schema names"
assert CATALOG not in ["hive_metastore", "system"], "Choose a writable Unity Catalog catalog in CATALOG"
PREFIX = f"`{CATALOG}`.`{SCHEMA}`"
FILES = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
def table(name):
    assert re.fullmatch(r"[a-z_]+", name)
    return f"{PREFIX}.`{name}`"
print("Catalog:", CATALOG, "Schema:", SCHEMA, "Volume path:", FILES)


In [ ]:
import pandas as pd
from mining_core import COLUMNS as EXPECTED_COLUMNS, clean_shifts
raw=pd.read_csv(f"{FILES}/mining_shifts_raw.csv",dtype=str,keep_default_na=False)
clean,rejected,counts=clean_shifts(raw)
print(counts)
assert counts == {'raw':610,'exact_duplicates_removed':4,'rejected':6,'accepted':600}


In [ ]:
from pyspark.sql import functions as F
# Explicit all-text schema keeps the original CSV values in Bronze.
bronze = spark.read.option("header", True).schema(",".join(f"{c} STRING" for c in EXPECTED_COLUMNS)).csv(f"{FILES}/mining_shifts_raw.csv")
bronze.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(table("bronze_shifts"))
# Explicit schemas avoid inference issues between runtimes.
silver_schema = "shift_id STRING, shift_date STRING, machine_id STRING, planned_hours DOUBLE, ore_hardness DOUBLE, feed_grade_pct DOUBLE, ambient_temp_c DOUBLE, prior_shift_availability DOUBLE, actual_tonnes DOUBLE, energy_kwh DOUBLE"
silver_rows = [(str(row[0]),str(row[1]),str(row[2]),*[float(z) for z in row[3:]]) for row in clean.itertuples(index=False,name=None)]
silver = spark.createDataFrame(silver_rows, silver_schema).withColumn("shift_date",F.to_date("shift_date"))
silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(table("silver_shifts"))
reject_schema = ",".join(f"{c} STRING" for c in EXPECTED_COLUMNS+["reject_reason"])
spark.createDataFrame(list(rejected.itertuples(index=False,name=None)),reject_schema).write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(table("rejected_shifts"))
assert spark.table(table("silver_shifts")).count() == 600
display(spark.table(table("rejected_shifts")))


## Checkpoint
Open Catalog → your schema → Tables. Bronze should contain 610 rows, Silver 600, rejected 6. Four exact duplicate copies were excluded, not classed as invalid. Rerunning this notebook overwrites only these named project tables. Never repoint the schema at a production schema.